# Jahresabschluss 2025
## EEG - Erneuerbare Energie Gemeinschaft

Dieses Notebook generiert automatisch den Jahresabschluss für 2025 aus der Django-Datenbank.

In [ ]:
import os
import sys
import django
from datetime import date

venv_path = '/home/martin/Workspace/Energiegemeinschaft/.venv/lib/python3.12/site-packages'
middleware_path = '/home/martin/Workspace/Energiegemeinschaft/middleware/eeg'

if venv_path not in sys.path:
    sys.path.insert(0, venv_path)
if middleware_path not in sys.path:
    sys.path.insert(0, middleware_path)

os.chdir('/home/martin/Workspace/Energiegemeinschaft/middleware/eeg')

os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'eeg.settings')

from django.conf import settings
if not settings.configured:
    django.setup()

In [ ]:
from accounting.models import Booking, BookingLabel
from django.db.models import Sum, Count, Q

import pandas as pd
import numpy as np

In [ ]:
YEAR = 2025
start_year = date(YEAR, 1, 1)
bookings_2025 = Booking.objects.filter(
    Q(payment_date__year=YEAR) | (Q(payment_date__isnull=True) & Q(booking_date__year=YEAR))
).order_by('booking_date', 'booking_reference')

prior_bookings = Booking.objects.filter(
    Q(payment_date__lt=start_year) | (Q(payment_date__isnull=True) & Q(booking_date__lt=start_year))
)
prior_balance = prior_bookings.aggregate(total=Sum('amount'))['total'] or 0

print(f'Anzahl Buchungen {YEAR}: {bookings_2025.count()}')
total_income = bookings_2025.filter(amount__gt=0).aggregate(total=Sum('amount'))['total'] or 0
total_expenses = bookings_2025.filter(amount__lt=0).aggregate(total=Sum('amount'))['total'] or 0
net_balance = total_income + total_expenses
final_balance = prior_balance + net_balance

print(f'Einnahmen: {total_income:,.2f} EUR')
print(f'Ausgaben: {total_expenses:,.2f} EUR')
print(f'Saldo {YEAR}: {net_balance:,.2f} EUR')
print(f'Kontostand vor {YEAR}: {prior_balance:,.2f} EUR')
print(f'Finaler Kontostand: {final_balance:,.2f} EUR')

In [ ]:
data = []
for booking in bookings_2025:
    effective_date = booking.payment_date if booking.payment_date else booking.booking_date
    labels = booking.labels.all()
    row = {
        'Buchungsdatum': booking.booking_date,
        'Zahlungsdatum': booking.payment_date,
        'Valutadatum': booking.value_date,
        'Partnername': booking.partner_name or '',
        'Partner IBAN': booking.partner_iban or '',
        'Buchungs-Details': booking.booking_details or '',
        'Betrag': float(booking.amount),
        'Währung': booking.currency or '',
        'Buchungsreferenz': booking.booking_reference or '',
        'Labels': ', '.join([label.label for label in labels]),
        'Label Count': labels.count(),
        'Jahr': effective_date.year,
    }
    data.append(row)

df = pd.DataFrame(data)
df['Buchungsdatum'] = pd.to_datetime(df['Buchungsdatum'])
df['Zahlungsdatum'] = pd.to_datetime(df['Zahlungsdatum'])
df['Valutadatum'] = pd.to_datetime(df['Valutadatum'])
df['Monat'] = df['Zahlungsdatum'].fillna(df['Buchungsdatum']).dt.month
df['Monatsname'] = df['Zahlungsdatum'].fillna(df['Buchungsdatum']).dt.strftime('%B')

print(f'DataFrame Shape: {df.shape}')
df.head()

In [ ]:
partner_summary = df.groupby('Partnername').agg({'Betrag': ['count', 'sum']}).round(2)
partner_summary.columns = ['Anzahl', 'Summe']
partner_summary = partner_summary.sort_values('Summe', ascending=False)
partner_summary

In [ ]:
month_order = ['Januar', 'Februar', 'März', 'April', 'Mai', 'Juni', 
              'Juli', 'August', 'September', 'Oktober', 'November', 'Dezember']
monthly_summary = df.groupby('Monatsname').agg({'Betrag': ['count', 'sum']}).round(2)
monthly_summary.columns = ['Anzahl', 'Summe']
monthly_summary = monthly_summary.reindex([m for m in month_order if m in monthly_summary.index])
monthly_summary

In [ ]:
labeled_df = df[df['Labels'] != ''].copy()
if not labeled_df.empty:
    labeled_df['Label'] = labeled_df['Labels'].str.split(', ')
    exploded = labeled_df.explode('Label')
    label_summary = exploded.groupby('Label').agg({'Betrag': ['count', 'sum']}).round(2)
    label_summary.columns = ['Anzahl', 'Summe']
    label_summary = label_summary.sort_values('Summe', ascending=False)
    display(label_summary)
else:
    print('Keine Buchungen mit Labels')

In [ ]:
income_df = df[df['Betrag'] > 0].sort_values('Betrag', ascending=False)
income_df[['Buchungsdatum', 'Partnername', 'Betrag', 'Buchungs-Details']].head(10)

In [ ]:
expenses_df = df[df['Betrag'] < 0].sort_values('Betrag', ascending=True)
expenses_df[['Buchungsdatum', 'Partnername', 'Betrag', 'Buchungs-Details']].head(10)

In [ ]:
output_dir = '/home/martin/Workspace/Energiegemeinschaft/notebooks/finance/Jahresabschluss/2025'
os.makedirs(output_dir, exist_ok=True)
output_file = os.path.join(output_dir, f'{YEAR}-01-01_{YEAR}-12-31.xlsx')

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    df.to_excel(writer, sheet_name='Alle Buchungen', index=False)
    summary_df = pd.DataFrame({
        'Kategorie': ['Einnahmen', 'Ausgaben', 'Saldo 2025', 'Kontostand vor 2025', 'Finaler Kontostand'],
        'Betrag': [total_income, total_expenses, net_balance, prior_balance, final_balance]
    })
    summary_df.to_excel(writer, sheet_name='Zusammenfassung', index=False)
    partner_summary.to_excel(writer, sheet_name='Nach Partner', index=True)
    monthly_summary.to_excel(writer, sheet_name='Nach Monat', index=True)
    if not labeled_df.empty:
        label_summary.to_excel(writer, sheet_name='Nach Labels', index=True)
    income_df.to_excel(writer, sheet_name='Einnahmen', index=False)
    expenses_df.to_excel(writer, sheet_name='Ausgaben', index=False)
print(f'Jahresabschluss wurde in "{output_file}" gespeichert!')

## Jahresabschluss 2025 - Fertig!

Der Jahresabschluss wurde erfolgreich generiert.